# Chapter 5 — Bayesian Estimation
## Figure 5.1: Prior Distributions · Figure 5.2: MCMC Convergence · Section 5.15: BVAR Empirical Example

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

> *"We all disagree, I think we should disagree, yeah"* — The Strokes, *Is This It?*  
> *"Bayes's theorem predicts that the Bayesians will win."* — Nate Silver

This notebook covers all figures and the empirical example from Chapter 5.

| Section | Content | Figures |
|---|---|---|
| **5.3** | Prior Distributions: Normal, Inverse Gamma, Beta | Figure 5.1 |
| **5.5** | MCMC Convergence Diagnostics: well-tuned vs poorly-tuned RWMH | Figure 5.2 |
| **5.15** | BVAR Empirical Example: U.S. monetary policy, Minnesota prior | Figures 5.3 – 5.6 |

**Key concepts introduced:**
- Bayes' theorem: posterior ∝ likelihood × prior
- Prior choice: conjugate priors (Normal-Inverse Wishart), informativeness
- MCMC: Random Walk Metropolis-Hastings, acceptance rate tuning, convergence diagnostics (ESS, Geweke)
- Minnesota prior: dummy-observation implementation, hyperparameter interpretation
- Posterior simulation: direct draws from Normal-Inverse Wishart (no MCMC needed for conjugate BVAR)
- Bayesian IRFs: credible intervals vs frequentist confidence intervals
- Prior sensitivity analysis

**Design conventions:** 🎛️ = tunable parameter cell · **Header** = why this step is needed · inline `#` = what each line does

---

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.linalg import cholesky
import warnings, os
warnings.filterwarnings('ignore')
np.random.seed(42)

OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Shared plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 9,
})

# Colour palette — consistent across all five figures
PRIOR_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
C_GOOD  = '#2171B5'   # well-tuned chain: blue
C_BAD   = '#CB181D'   # poorly-tuned chain: red
C_BVAR  = '#E94F37'   # BVAR posterior median: red
C_OLS   = '#2E86AB'   # OLS estimate: blue
C_BANDS = '#A23B72'   # credible interval fill: purple

print('All packages loaded.')

---
# Figure 5.1 — Common Prior Distributions
*Textbook reference: Section 5.3*

## Background

In Bayesian inference, the **prior distribution** $p(\theta)$ encodes beliefs about parameters *before* observing data. The key design choice is:

1. **Family**: which distributional form captures the support and shape of the parameter?
2. **Hyperparameters**: how tight or diffuse should the prior be?

Three families are used throughout the textbook:

| Distribution | Support | Primary use | Conjugate for |
|---|---|---|---|
| **Normal** $N(\mu_0, \sigma_0^2)$ | $\mathbb{R}$ | Location parameters, VAR coefficients | Gaussian likelihood (known variance) |
| **Inverse Gamma** $IG(\alpha, \beta)$ | $\mathbb{R}_{>0}$ | Variance parameters $\sigma^2$ | Gaussian likelihood (unknown variance) |
| **Beta** $\text{Beta}(\alpha, \beta)$ | $(0,1)$ | Probabilities, regime transition probs | Binomial/Bernoulli likelihood |

A prior is **conjugate** when the posterior belongs to the same family as the prior — this enables analytical posterior computation without MCMC, as exploited in the Minnesota-prior BVAR of Section 5.15.

In [ ]:
# ── Figure 5.1: Common Prior Distributions ───────────────────────────────────
# 3-panel figure replicating textbook Figure 5.1.
# Each panel illustrates how hyperparameters control shape and tightness.

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Figure 5.1: Common Prior Distributions for Bayesian Estimation',
             fontsize=13, fontweight='bold')

# ── Panel (a): Normal Prior ───────────────────────────────────────────────────
# Used for VAR coefficients in the Minnesota prior.
# Key lesson: σ² controls the spread (informativeness) of the prior.
# Smaller σ² = tighter prior = more shrinkage toward the prior mean.
ax = axes[0]
x_norm = np.linspace(-6, 6, 500)

normal_params = [
    (0, 0.5, r'$N(0,\, 0.5^2)$ — Tight'),
    (0, 1.0, r'$N(0,\, 1^2)$ — Moderate'),
    (0, 2.0, r'$N(0,\, 2^2)$ — Diffuse'),
    (1, 1.0, r'$N(1,\, 1^2)$ — Shifted mean'),
]
for i, (mu0, sig0, label) in enumerate(normal_params):
    ax.plot(x_norm, stats.norm.pdf(x_norm, mu0, sig0),
            color=PRIOR_COLORS[i], linewidth=2.5, label=label)

ax.set_xlabel(r'$\theta$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('(a) Normal Prior\nLocation parameters', fontsize=11)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.9)
ax.set_xlim(-6, 6);  ax.set_ylim(0, 0.85)

# ── Panel (b): Inverse Gamma Prior ───────────────────────────────────────────
# Used for residual variance σ² in BVAR.
# scipy parameterisation: IG(α, β) = stats.invgamma(a=α, scale=β)
# Mean = β/(α-1) for α>1.  Smaller α (fewer pseudo-observations) = heavier tail.
ax = axes[1]
x_ig = np.linspace(0.01, 5, 500)

ig_params = [
    (3, 2, r'$IG(3,\,2)$ — Moderate'),
    (2, 1, r'$IG(2,\,1)$ — Diffuse'),
    (1, 1, r'$IG(1,\,1)$ — Heavy tail'),
    (8, 3, r'$IG(8,\,3)$ — Informative'),
]
for i, (alpha, beta, label) in enumerate(ig_params):
    # stats.invgamma(a=α, scale=β) corresponds to IG(α, β) with pdf ∝ x^{-α-1} exp(-β/x)
    ax.plot(x_ig, stats.invgamma.pdf(x_ig, a=alpha, scale=beta),
            color=PRIOR_COLORS[i], linewidth=2.5, label=label)

ax.set_xlabel(r'$\sigma^2$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('(b) Inverse Gamma Prior\nVariance parameters', fontsize=11)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.9)
ax.set_xlim(0, 5);  ax.set_ylim(0, 4)

# ── Panel (c): Beta Prior ─────────────────────────────────────────────────────
# Used for regime transition probabilities in MS-VAR (Chapter 10) and
# probabilities of events in Bayesian model selection.
# α=β=1 is the uniform prior (no information).  α=β>1 concentrates around 0.5.
ax = axes[2]
x_beta = np.linspace(0.001, 0.999, 500)

beta_params = [
    (1,   1,   r'$\mathrm{Beta}(1,1)$ — Uniform'),
    (0.5, 0.5, r'$\mathrm{Beta}(0.5,0.5)$ — U-shaped'),
    (2,   5,   r'$\mathrm{Beta}(2,5)$ — Right-skewed'),
    (5,   2,   r'$\mathrm{Beta}(5,2)$ — Left-skewed'),
    (5,   5,   r'$\mathrm{Beta}(5,5)$ — Symmetric'),
]
for i, (a, b, label) in enumerate(beta_params):
    ax.plot(x_beta, stats.beta.pdf(x_beta, a, b),
            color=PRIOR_COLORS[i], linewidth=2.5, label=label)

ax.set_xlabel(r'$p$', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('(c) Beta Prior\nProbabilities & proportions', fontsize=11)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.9)
ax.set_xlim(0, 1);  ax.set_ylim(0, 4)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig51_prior_distributions.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

**What to look for:**
- **Panel (a):** The tight Normal prior ($\sigma=0.5$) concentrates probability in a narrow band around zero — with λ₁ = 0.2 in the Minnesota prior, most cross-variable VAR coefficients have priors of roughly this width. As σ increases, the prior becomes uninformative and the posterior is driven almost entirely by the data.
- **Panel (b):** The IG(1,1) case has a very heavy right tail — it assigns substantial probability to large variance values. In practice, empirical Bayes (Giannone et al. 2015) chooses the IG hyperparameters to match the residual variance of AR(1) regressions, making the prior adaptive to each variable's scale.
- **Panel (c):** The U-shaped Beta(0.5, 0.5) is a *non-informative* Jeffreys prior for a probability parameter. The Beta(5,5) is appropriate when we have strong prior belief that the probability is near 0.5 — for example, the prior on a regime-switching transition probability when we expect regimes to be roughly equally persistent.

---
# Figure 5.2 — MCMC Convergence Diagnostics
*Textbook reference: Section 5.5*

## Background

When the posterior is not analytically tractable (e.g., non-conjugate priors, nonlinear models), we use **Markov Chain Monte Carlo (MCMC)** to simulate draws from $p(\theta | y)$. The **Random Walk Metropolis-Hastings (RWMH)** algorithm:

1. Start at $\theta^{(0)}$
2. Propose: $\theta^* = \theta^{(t-1)} + c \cdot \varepsilon$, $\varepsilon \sim N(0, I)$
3. Accept with probability: $\alpha = \min\left(1, \frac{p(\theta^* | y)}{p(\theta^{(t-1)} | y)}\right)$
4. Set $\theta^{(t)} = \theta^*$ if accepted, else $\theta^{(t)} = \theta^{(t-1)}$

The scaling constant $c$ controls the **proposal step size**:
- Too large $c$: most proposals fall in low-density regions → almost always rejected → chain gets stuck
- Too small $c$: proposals are tiny steps → almost always accepted → chain moves slowly (high autocorrelation)
- Goldilocks: acceptance rate ≈ 23–44% (Roberts, Gelman & Gilks 1997)

**Convergence diagnostics:**
- **Trace plot**: should look like a *fuzzy caterpillar* — rapid mixing without visible trends or stickiness
- **ACF**: autocorrelation should decay to zero quickly. Slow decay = high serial correlation = low effective sample size
- **Effective Sample Size (ESS)**: $\text{ESS} = T / (1 + 2 \sum_{k=1}^\infty \rho_k)$. Measures how many independent draws the chain is equivalent to
- **Geweke diagnostic**: tests whether the mean of the first 10% of the chain equals the mean of the last 50%. Under convergence, the $Z$-score should be within ±1.96

## Data: pre-generated chains

The chains are generated by `mcmc_generate_chains.py` and saved as CSV files. The target distribution is bivariate Normal $\mathcal{N}(\mathbf{0}, \Sigma)$ with $\rho = 0.7$. Two chains are compared:

| Chain | Scale $c$ | Expected behaviour |
|---|---|---|
| **Well-tuned** | $c = 1.70$ | Acceptance rate ≈ 26%, rapid mixing |
| **Poorly-tuned** | $c = 0.05$ | Acceptance rate ≈ 96%, very slow mixing |

In [ ]:
# 🎛️ FIGURE 5.2 PARAMETERS
PATH_CHAIN_GOOD = 'mcmc_chain_good.csv'
PATH_CHAIN_BAD  = 'mcmc_chain_bad.csv'
MAX_LAG         = 100    # maximum lag for ACF plots
FIRST_FRAC      = 0.10   # Geweke: fraction of chain used as 'early' segment
LAST_FRAC       = 0.50   # Geweke: fraction of chain used as 'late' segment

In [ ]:
# ── Load pre-generated chains ─────────────────────────────────────────────────
# The CSVs have header 'theta1,theta2' → skiprows=1 skips it
chain_good = np.loadtxt(PATH_CHAIN_GOOD, delimiter=',', skiprows=1)
chain_bad  = np.loadtxt(PATH_CHAIN_BAD,  delimiter=',', skiprows=1)
n_iter     = len(chain_good)
mu_target  = np.array([0.0, 0.0])   # true posterior mean (known for this toy example)

# Acceptance rate: count iterations where the chain moved (values changed)
ar_good = np.mean(np.any(np.diff(chain_good, axis=0) != 0, axis=1))
ar_bad  = np.mean(np.any(np.diff(chain_bad,  axis=0) != 0, axis=1))

print(f'Chains loaded: T = {n_iter:,} iterations, d = {chain_good.shape[1]} parameters')
print(f'Well-tuned:   acceptance rate = {ar_good:.1%}  (target: 23–44%)')
print(f'Poorly-tuned: acceptance rate = {ar_bad:.1%}  (too high → slow mixing)')

In [ ]:
# ── Compute ACF and ESS ───────────────────────────────────────────────────────

def compute_acf(x, max_lag):
    """
    Compute sample autocorrelation function of a univariate time series x.

    ACF(k) = Cov(x_t, x_{t-k}) / Var(x_t)

    We use the direct definition (not FFT-based) for clarity.
    ACF(0) = 1 by construction.
    """
    n   = len(x)
    xc  = x - x.mean()         # centre the series
    var = np.var(xc)            # Var(x)
    acf = np.zeros(max_lag + 1)
    for k in range(max_lag + 1):
        # Cov(x_t, x_{t-k}) = mean of product of overlapping segments
        acf[k] = np.mean(xc[:n-k] * xc[k:]) / var if var > 0 else 0
    return acf

def compute_ess(acf_vals, n):
    """
    Estimate Effective Sample Size (ESS) from the ACF.

    ESS = n / τ  where  τ = 1 + 2 Σ_{k=1}^∞ ρ_k
    τ is the integrated autocorrelation time.

    We truncate the sum at the first negative ACF value (initial monotone sequence estimator).
    This avoids noise inflation from including large-lag ACF estimates that are near zero.
    """
    tau = 1.0
    for k in range(1, len(acf_vals)):
        if acf_vals[k] < 0:
            break       # stop at first negative ACF (monotone sequence truncation)
        tau += 2 * acf_vals[k]
    return n / tau

acf_good = compute_acf(chain_good[:, 0], MAX_LAG)
acf_bad  = compute_acf(chain_bad[:,  0], MAX_LAG)
ess_good = compute_ess(acf_good, n_iter)
ess_bad  = compute_ess(acf_bad,  n_iter)

print(f'Well-tuned:   ESS = {ess_good:.0f} / {n_iter:,}  ({100*ess_good/n_iter:.1f}% efficiency)')
print(f'Poorly-tuned: ESS = {ess_bad:.0f} / {n_iter:,}  ({100*ess_bad/n_iter:.2f}% efficiency)')
print(f'\nThe poorly-tuned chain has {ess_good/ess_bad:.0f}× lower efficiency.')

In [ ]:
# ── Geweke diagnostic ─────────────────────────────────────────────────────────
# The Geweke (1992) test compares the posterior mean of the FIRST 10% of the chain
# to the mean of the LAST 50% of the chain. Under convergence, these should be equal.
# The test statistic is a Z-score using spectral variance estimates for each segment.
#
# We also compute a ROLLING Geweke: for each starting point of the 'late' segment,
# we compute the Z-score. If the chain has converged, all Z-scores should lie in [-1.96, 1.96].

def spectral_variance(x):
    """
    Estimate the spectral variance (variance of the sample mean) of x at frequency zero.

    This is Var(x̄) = (1/n) Σ_{k=-(n-1)}^{n-1} w_k γ_k

    where w_k = 1 - k/(m+1) is the Bartlett kernel weight and m = sqrt(n).
    This bandwidth rule follows the textbook's implementation.
    """
    n  = len(x)
    m  = int(np.sqrt(n))   # Bartlett bandwidth
    xc = x - x.mean()
    s  = np.var(xc)        # γ_0 term
    for k in range(1, m + 1):
        w      = 1 - k / (m + 1)              # Bartlett weight (triangular taper)
        gamma_k = np.mean(xc[:n-k] * xc[k:])  # autocovariance at lag k
        s      += 2 * w * gamma_k
    return s / n   # divide by n: this is the variance of the MEAN

def geweke_z(chain_1d, first_frac=0.10, last_frac=0.50):
    """Standard Geweke Z-score: first_frac vs last_frac of chain."""
    n        = len(chain_1d)
    seg_a    = chain_1d[:int(first_frac * n)]   # first 10%
    seg_b    = chain_1d[n - int(last_frac * n):]  # last 50%
    var_a    = spectral_variance(seg_a)           # spectral variance of mean_a
    var_b    = spectral_variance(seg_b)           # spectral variance of mean_b
    denom    = np.sqrt(var_a + var_b)
    return (seg_a.mean() - seg_b.mean()) / denom if denom > 0 else 0.0

def geweke_rolling(chain_1d, first_frac=0.10, window_starts=None):
    """Compute Geweke Z-score for each possible start of the 'late' segment."""
    n     = len(chain_1d)
    seg_a = chain_1d[:int(first_frac * n)]
    var_a = spectral_variance(seg_a)
    z_scores = np.zeros(len(window_starts))
    for i, start in enumerate(window_starts):
        seg_b    = chain_1d[start:]              # 'late' segment starts at 'start'
        var_b    = spectral_variance(seg_b)
        denom    = np.sqrt(var_a + var_b)
        z_scores[i] = (seg_a.mean() - seg_b.mean()) / denom if denom > 0 else 0
    return z_scores

# Rolling window: start points from 20% to 80% of chain
window_starts  = np.arange(int(0.20 * n_iter), int(0.80 * n_iter), 50)
gw_good_roll   = geweke_rolling(chain_good[:, 0], FIRST_FRAC, window_starts)
gw_bad_roll    = geweke_rolling(chain_bad[:,  0], FIRST_FRAC, window_starts)
z_good         = geweke_z(chain_good[:, 0], FIRST_FRAC, LAST_FRAC)
z_bad          = geweke_z(chain_bad[:,  0], FIRST_FRAC, LAST_FRAC)

p_good = 2 * (1 - stats.norm.cdf(abs(z_good)))
p_bad  = 2 * (1 - stats.norm.cdf(abs(z_bad)))
print(f'Geweke Z (first {int(FIRST_FRAC*100)}% vs last {int(LAST_FRAC*100)}%):')
print(f'  Well-tuned:   Z = {z_good:.3f},  p = {p_good:.3f}  → {"✓ converged" if abs(z_good) < 1.96 else "✗ NOT converged"}')
print(f'  Poorly-tuned: Z = {z_bad:.3f},  p = {p_bad:.3f}  → {"✓ converged" if abs(z_bad) < 1.96 else "✗ NOT converged"}')

In [ ]:
# ── Figure 5.2: MCMC Convergence Diagnostics (3-row × 2-col) ─────────────────
# Row 1: Trace plots     (did the chain mix?)
# Row 2: ACF plots       (how correlated are adjacent draws?)
# Row 3: Geweke rolling  (has the chain converged to stationarity?)

fig = plt.figure(figsize=(14, 13))
gs  = GridSpec(3, 2, hspace=0.45, wspace=0.28)
fig.suptitle(
    'Figure 5.2: MCMC Convergence Diagnostics — Random Walk Metropolis-Hastings\n'
    r'Target: $\mathcal{N}(\mathbf{0},\,\Sigma)$ with $\rho=0.7$; '
    f'$T = {n_iter:,}$ iterations',
    fontsize=13, fontweight='bold', y=1.01
)

# ── Row 1: Trace plots ────────────────────────────────────────────────────────
# A well-converged chain should explore the whole support rapidly (hairy caterpillar).
# A poorly-tuned chain gets stuck and moves very slowly.
for col, (chain, color, label, c_val) in enumerate([
    (chain_good, C_GOOD, 'Well-Tuned',   '1.70'),
    (chain_bad,  C_BAD,  'Poorly-Tuned', '0.05'),
]):
    ar   = ar_good if col == 0 else ar_bad
    ess_ = ess_good if col == 0 else ess_bad
    ax = fig.add_subplot(gs[0, col])
    ax.plot(chain[:, 0], color=color, linewidth=0.3, alpha=0.85)
    ax.axhline(mu_target[0], color='black', linestyle='--', linewidth=1.0, alpha=0.5,
               label='True mean $\\mu_1 = 0$')
    ax.set_title(f'({'ab'[col]}) {label} Trace Plot ($c = {c_val}$)\n'
                 f'Acceptance rate: {ar:.1%}  |  ESS: {ess_:.0f}')
    ax.set_xlabel('Iteration');  ax.set_ylabel(r'$\theta_1$')
    ax.set_xlim(0, n_iter);      ax.set_ylim(-4, 4)
    ax.legend(fontsize=8, loc='upper right')

# ── Row 2: ACF plots ──────────────────────────────────────────────────────────
# Bar chart of ACF(k) for k = 0, ..., 100.
# Gray dashed lines = 95% critical value 1.96/sqrt(T) under IID null.
crit = 1.96 / np.sqrt(n_iter)   # asymptotic 95% CI for IID series
for col, (acf_vals, color, label) in enumerate([
    (acf_good, C_GOOD, '(c) Well-Tuned Autocorrelation Function'),
    (acf_bad,  C_BAD,  '(d) Poorly-Tuned Autocorrelation Function'),
]):
    ax = fig.add_subplot(gs[1, col])
    ax.bar(range(MAX_LAG + 1), acf_vals, width=1.0, color=color, alpha=0.7, edgecolor='none')
    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( crit, color='gray', linestyle='--', linewidth=0.8, alpha=0.6,
                label='95% CI (IID)')
    ax.axhline(-crit, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_title(label)
    ax.set_xlabel('Lag');  ax.set_ylabel('ACF')
    ax.set_xlim(0, MAX_LAG);  ax.set_ylim(-0.1, 1.05)
    ax.legend(fontsize=8)

# ── Row 3: Geweke rolling Z-score ─────────────────────────────────────────────
# We slide the start of the 'late' segment across 20%–80% of the chain.
# A converged chain should have Z-scores inside the ±1.96 band throughout.
for col, (z_roll, z_scalar, color, label) in enumerate([
    (gw_good_roll, z_good, C_GOOD, '(e) Well-Tuned Geweke Diagnostic'),
    (gw_bad_roll,  z_bad,  C_BAD,  '(f) Poorly-Tuned Geweke Diagnostic'),
]):
    ax = fig.add_subplot(gs[2, col])
    ax.plot(window_starts, z_roll, color=color, linewidth=1.5)
    ax.axhline(0,    color='black', linewidth=0.8)
    ax.axhline( 1.96, color='gray', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.axhline(-1.96, color='gray', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.fill_between(window_starts, -1.96, 1.96, alpha=0.07, color='gray',
                    label='95% acceptance region')
    ax.set_title(f'{label}\n$Z = {z_scalar:.2f}$,  $p = {2*(1-stats.norm.cdf(abs(z_scalar))):.3f}$')
    ax.set_xlabel('Start of comparison window')
    ax.set_ylabel('Geweke $Z$-score')
    ax.set_xlim(window_starts[0], window_starts[-1])
    ax.set_ylim(-5, 5)
    ax.legend(fontsize=8)
    ax.text(window_starts[-1] - 300, 2.3, '95% critical values', fontsize=8,
            color='gray', ha='right', style='italic')

path = os.path.join(OUT_DIR, 'fig52_mcmc_convergence.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

**What to look for:**
- **Trace plots (a,b):** The well-tuned chain (blue) rapidly explores $[-3, 3]$ with no visible trends — it looks like a fuzzy band (the "hairy caterpillar"). The poorly-tuned chain (red) moves in tiny steps; you can see extended flat segments where it gets stuck. Notice the much smaller movement range despite the same number of iterations.
- **ACF (c,d):** The well-tuned chain's ACF drops to within the 95% CI band by lag 10–15. The poorly-tuned chain's ACF is still near 1.0 at lag 100 — meaning each draw is essentially the same as the last 100 draws. This translates into the ESS being a tiny fraction of T.
- **Geweke (e,f):** The well-tuned Z-score is near zero throughout (convergence). The poorly-tuned chain shows Z-scores far outside ±1.96, indicating the early and late parts of the chain have very different means — the chain has not converged to its stationary distribution.

---
# Section 5.15 — BVAR Empirical Example: U.S. Monetary Policy
*Textbook reference: Section 5.15, Figures 5.3 – 5.6*

## Background

We revisit the 3-variable monetary policy VAR from Chapter 4, now estimated in a **Bayesian** framework with the **Minnesota prior**. This lets us compare:
- OLS (unrestricted, maximum likelihood)
- BVAR with Minnesota prior (shrinkage toward random walk)

### Minnesota Prior via Dummy Observations

The Minnesota prior implemented via dummy observations appends artificial data $(Y_d, X_d)$ to the actual data before OLS estimation. The two sets of dummies are:

**Dummy type 1 — own-lag shrinkage** (one row per variable $i$, lag $l$):
$$y_{il} = \frac{\hat{\sigma}_i}{\lambda_1 l^{\lambda_3}} e_i', \quad x_{il} = \frac{\hat{\sigma}_i}{\lambda_1 l^{\lambda_3}} e_{i,\text{lag-}l}'$$
This centers $A_l^{ii} = 1$ (random walk prior on own lags), with tightness $\lambda_1 / l^{\lambda_3}$.

**Dummy type 2 — covariance prior** ($K$ rows for the initial levels):
$$y_i = \hat{\sigma}_i e_i', \quad x_i = \text{diag}(\hat{\sigma}_1, ..., \hat{\sigma}_K)$$
This implements the sum-of-coefficients restriction (Sims 1993), shrinking the residual covariance toward a diagonal matrix scaled by AR(1) residual variances.

The **posterior** is analytically available: given the conjugate Normal-Inverse Wishart prior implied by these dummies, posterior draws are obtained directly (no MCMC needed):
$$\Sigma^{(j)} \sim IW(\nu_T, S_T) \qquad \text{then} \qquad \text{vec}(A^{(j)}) | \Sigma^{(j)} \sim N(\alpha_T, \Sigma^{(j)} \otimes V_T^{-1})$$

**Hyperparameters:** $\lambda_1 = 0.2$ (overall tightness), $\lambda_2 = 0.5$ (cross-variable shrinkage, not used separately here — absorbed into $\lambda_1$), $\lambda_3 = 1$ (lag decay).

In [ ]:
# 🎛️ SECTION 5.15 PARAMETERS
PATH_GDP   = 'GDPC1.xlsx'
PATH_DEFL  = 'GDPDEF.xlsx'
PATH_FF    = 'FEDFUNDS.xlsx'
S15_START  = '1970-01-01'
S15_END    = '2007-12-31'
P          = 4        # VAR lag order
N_DRAWS    = 2000     # posterior draws (no burn-in needed for conjugate posteriors)
H          = 20       # IRF horizon (quarters)

# Minnesota prior hyperparameters (baseline)
LAMBDA1    = 0.20     # overall tightness
LAMBDA3    = 1.00     # lag decay exponent

# Prior sensitivity grid
LAMBDA1_GRID = [0.05, 0.10, 0.20, 0.50, 1.00]

VAR_NAMES = ['GDP Growth', 'Inflation', 'Federal Funds Rate']
COLORS_5  = ['#1b7837', '#762a83', '#b2182b', '#2166ac', '#d6604d']  # sensitivity grid

In [ ]:
# ── Load FRED data ────────────────────────────────────────────────────────────
def load_fred_q(path, name):
    """Load quarterly FRED Excel file to date-indexed DataFrame."""
    d = pd.read_excel(path, sheet_name='Quarterly', header=0)
    d.columns = ['date', name]
    d['date'] = pd.to_datetime(d['date']).dt.to_period('Q').dt.to_timestamp('Q')
    return d.set_index('date')

gdp  = load_fred_q(PATH_GDP,  'GDPC1')
defl = load_fred_q(PATH_DEFL, 'GDPDEF')

# Fed funds: monthly → quarterly average
ff_raw = pd.read_excel(PATH_FF, sheet_name='Monthly', header=0)
ff_raw.columns = ['date', 'FEDFUNDS']
ff_raw['date'] = pd.to_datetime(ff_raw['date'])
ff_q = ff_raw.set_index('date').resample('QE').mean()

# 400 × Δln = annualised quarter-on-quarter growth rates
gdp_g = 400 * np.log(gdp['GDPC1']   / gdp['GDPC1'].shift(1))
inf_  = 400 * np.log(defl['GDPDEF'] / defl['GDPDEF'].shift(1))

# Normalise all series to quarter-end dates before merging
gdp_g.index = gdp_g.index.to_period('Q').to_timestamp('Q')
inf_.index  = inf_.index.to_period('Q').to_timestamp('Q')

data = pd.DataFrame({'gdp': gdp_g, 'inf': inf_, 'ff': ff_q['FEDFUNDS']}).dropna()
data = data.loc[S15_START:S15_END]

first_q = f"{data.index[0].year}:Q{data.index[0].quarter}"
last_q  = f"{data.index[-1].year}:Q{data.index[-1].quarter}"

print(f'Sample: {first_q} – {last_q}  |  T = {len(data)}')
print(data.describe().round(3))

### Step 1 — OLS benchmark

In [ ]:
# ── OLS VAR(p) estimation ─────────────────────────────────────────────────────
def ols_var(Y, p):
    """
    Estimate VAR(p) by OLS.  Returns (T_eff, B_hat, U, Sigma_u, X).
    B_hat: (1 + K*p) × K  [constant; A_1; ...; A_p]
    Sigma_u: K × K residual covariance (df-corrected)
    """
    T, K  = Y.shape
    T_eff = T - p
    Y_dep = Y[p:]
    X = np.ones((T_eff, 1))
    for lag in range(1, p + 1):
        X = np.hstack([X, Y[p - lag: T - lag]])
    B_hat = np.linalg.lstsq(X, Y_dep, rcond=None)[0]
    U     = Y_dep - X @ B_hat
    Sigma = (U.T @ U) / (T_eff - K * p - 1)
    return T_eff, B_hat, U, Sigma, X

Y = data.values
K = Y.shape[1]

T_eff, B_ols, U_ols, Sigma_ols, X_ols = ols_var(Y, P)

# AR(1) residual standard deviations — used to scale Minnesota dummies
sigma_hat = np.array([np.std(np.diff(Y[:, k])) for k in range(K)])

print(f'OLS VAR({P}): T_eff = {T_eff}  |  K = {K}  |  params per eq = {1 + K*P}')
print(f'\nAR(1) residual σ̂ (for Minnesota scaling):')
for k, name in enumerate(VAR_NAMES):
    print(f'  {name:<25}: {sigma_hat[k]:.4f}')

### Step 2 — Minnesota prior via dummy observations

In [ ]:
# ── Minnesota prior via dummy observations ────────────────────────────────────

def minnesota_dummies(sigma_hat, K, p, lambda1, lambda3=1.0):
    """
    Construct dummy observations implementing the Minnesota prior.

    The two blocks of dummies are:

    Block 1 — Lag-coefficient prior (K*p rows):
      For each variable i and lag l, one row with:
        Y_d[row, i] = sigma_i / (lambda1 * l^lambda3)
        X_d[row, i-th lag-l position] = sigma_i / (lambda1 * l^lambda3)
      This imposes A_{l,ii} ~ N(1_{l=1}, (lambda1 * l^lambda3)^2 / sigma_i^2)
      i.e. own-lag 1 shrunk toward 1 (random walk), higher lags toward 0.

    Block 2 — Residual covariance prior (K rows):
      One row per variable with Y_d[i, i] = sigma_i, X_d = diag(sigma).
      This shrinks the residual covariance toward diag(sigma^2).

    Parameters
    ----------
    sigma_hat : K-vector of AR(1) residual std devs (scale parameters)
    K         : number of variables
    p         : lag order
    lambda1   : overall tightness (smaller = more shrinkage)
    lambda3   : lag decay exponent (=1 means harmonic decay)

    Returns
    -------
    Y_d : (K*p + K) × K dummy dependent variable matrix
    X_d : (K*p + K) × (1 + K*p) dummy regressor matrix
    """
    n_d1 = K * p      # Block 1: K*p dummy rows
    n_d2 = K          # Block 2: K dummy rows
    n_d  = n_d1 + n_d2
    cols = 1 + K * p  # number of regressors (constant + K*p lags)

    Y_d = np.zeros((n_d, K))
    X_d = np.zeros((n_d, cols))

    row = 0
    # ── Block 1: lag-coefficient prior ───────────────────────────────────────
    for l in range(1, p + 1):           # for each lag l = 1, ..., p
        scale_l = lambda1 * (l ** lambda3)  # tightness at lag l (looser for higher lags)
        for i in range(K):               # for each variable i
            # The diagonal entry: Y_d[i] = sigma_i / scale_l
            Y_d[row, i] = sigma_hat[i] / scale_l
            # The corresponding X_d position: column 1 + (l-1)*K + i
            # (columns: constant | A_1 vectorised | A_2 vectorised | ... )
            col_idx = 1 + (l - 1) * K + i
            X_d[row, col_idx] = sigma_hat[i] / scale_l
            row += 1

    # ── Block 2: residual covariance prior ────────────────────────────────────
    for i in range(K):
        Y_d[row, i] = sigma_hat[i]    # Y_d = diag(sigma)
        # X_d for the covariance dummy: all lag-1 positions set to sigma
        for j in range(K):
            X_d[row, 1 + j] = sigma_hat[j]   # positions 1..K (lag 1)
        row += 1

    return Y_d, X_d


def bvar_posterior(Y, p, lambda1, lambda3=1.0):
    """
    Compute BVAR posterior parameters with Minnesota prior (dummy observations).

    The augmented OLS [Y_aug, X_aug] = [Y/Y_d, X/X_d] yields the posterior
    Normal-Inverse Wishart parameters:

      V_T^{-1} = X_aug' X_aug
      alpha_T  = V_T (X_aug' Y_aug)
      nu_T     = T_eff + n_d
      S_T      = (Y_aug - X_aug alpha_T)' (Y_aug - X_aug alpha_T)

    Returns posterior sufficient statistics + sigma_hat for dummies.
    """
    T, K     = Y.shape
    T_eff    = T - p
    Y_dep    = Y[p:]
    X = np.ones((T_eff, 1))
    for lag in range(1, p + 1):
        X = np.hstack([X, Y[p - lag: T - lag]])

    # AR(1) std devs as scale parameters for the dummies
    sigma_hat = np.array([np.std(np.diff(Y[:, k])) for k in range(K)])

    # Build dummy observations
    Y_d, X_d = minnesota_dummies(sigma_hat, K, p, lambda1, lambda3)
    n_d      = len(Y_d)

    # Augment actual data with dummies
    Y_aug = np.vstack([Y_dep, Y_d])
    X_aug = np.vstack([X,     X_d])

    # Posterior precision matrix V_T^{-1} = X_aug' X_aug
    XTX  = X_aug.T @ X_aug          # (1 + K*p) × (1 + K*p)
    XTY  = X_aug.T @ Y_aug          # (1 + K*p) × K
    V_T_inv = XTX                   # posterior precision
    V_T     = np.linalg.inv(XTX)    # posterior covariance of vec(A)

    # Posterior mean alpha_T = V_T (X_aug' Y_aug)
    alpha_T = V_T @ XTY             # (1 + K*p) × K  — posterior mean of A

    # Posterior IW scale matrix S_T
    resid   = Y_aug - X_aug @ alpha_T
    S_T     = resid.T @ resid       # K × K

    # Posterior degrees of freedom
    nu_T = T_eff + n_d              # total effective observations

    return alpha_T, V_T, S_T, nu_T, K, sigma_hat


alpha_T, V_T, S_T, nu_T, K, sig_hat = bvar_posterior(Y, P, LAMBDA1, LAMBDA3)

print(f'BVAR posterior computed (λ₁ = {LAMBDA1})')
print(f'nu_T = {nu_T}  (T_eff + n_dummies = {T_eff} + {nu_T - T_eff})')
print(f'\nPosterior mean vs OLS comparison (own lag-1 coefficients):')
print(f'{"Variable":<25}  {"OLS":>8}  {"BVAR":>8}  {"Shrinkage %":>12}')
print('-' * 58)
for k, name in enumerate(VAR_NAMES):
    ols_coef  = B_ols[1 + k, k]    # own lag-1: row 1+k (A_1), col k
    bvar_coef = alpha_T[1 + k, k]
    shrink    = 100 * abs(bvar_coef - ols_coef) / (abs(ols_coef) + 1e-10)
    print(f'{name:<25}  {ols_coef:>8.4f}  {bvar_coef:>8.4f}  {shrink:>11.1f}%')

### Step 3 — Posterior simulation and IRF computation

Because we use conjugate priors, each posterior draw is **independent** — no burn-in, no thinning, no convergence monitoring needed. The algorithm:
1. Draw $\Sigma^{(j)} \sim IW(\nu_T, S_T)$ via the Bartlett decomposition
2. Draw $\text{vec}(A^{(j)}) | \Sigma^{(j)} \sim N(\alpha_T,\, \Sigma^{(j)} \otimes V_T^{-1})$
3. Compute Cholesky $P^{(j)} = \text{chol}(\Sigma^{(j)})$ and IRF from companion matrix

In [ ]:
# ── Posterior simulation ──────────────────────────────────────────────────────

def draw_iw(nu, S):
    """
    Draw from Inverse Wishart IW(nu, S) via Bartlett decomposition.

    Algorithm (Odell & Feiveson 1966):
      1. Draw L = chol(S)^{-T}  (upper Cholesky inverse)
      2. Build lower-triangular A where A_ii ~ chi(nu-i) and A_ij ~ N(0,1) for j<i
      3. X = (L A)^{-1} (L A)^{-T} ~ IW(nu, S)
    """
    K = S.shape[0]
    L = np.linalg.cholesky(S)  # lower triangular L: S = L L'

    # Build lower-triangular Bartlett factor A
    A = np.zeros((K, K))
    for i in range(K):
        A[i, i] = np.sqrt(np.random.chisquare(nu - i))  # diagonal: chi-distributed
        for j in range(i):
            A[i, j] = np.random.randn()                  # off-diagonal: N(0,1)

    # B = L @ A  (lower triangular × lower triangular = lower triangular)
    B = L @ A
    # IW draw: Sigma = B^{-T} B^{-1} = (B B')^{-1}
    B_inv = np.linalg.inv(B)
    return B_inv @ B_inv.T


def compute_ma_coeff(B_hat, p, K, H):
    """Reduced-form MA coefficients Φ_0, ..., Φ_H via recursive formula."""
    B = B_hat[1:].reshape(p, K, K).transpose(0, 2, 1)
    Phi = np.zeros((H + 1, K, K))
    Phi[0] = np.eye(K)
    for h in range(1, H + 1):
        for j in range(min(h, p)):
            Phi[h] += B[j] @ Phi[h - j - 1]
    return Phi


def irf_cholesky(B_hat, Sigma, p, K, H):
    """Cholesky IRF: Θ_h = Φ_h P,  P = chol(Sigma, lower=True)."""
    P   = cholesky(Sigma, lower=True)
    Phi = compute_ma_coeff(B_hat, p, K, H)
    IRF = np.zeros((H + 1, K, K))
    for h in range(H + 1):
        IRF[h] = Phi[h] @ P
    return IRF


print(f'Drawing {N_DRAWS} posterior samples from Normal-Inverse Wishart...')
IRF_draws = np.zeros((N_DRAWS, H + 1, K, K))   # store all IRF draws
n_fail = 0

for j in range(N_DRAWS):
    try:
        # Step 1: draw Sigma from IW posterior
        Sigma_j = draw_iw(nu_T, S_T)

        # Step 2: draw A | Sigma from Normal posterior
        # vec(A) | Sigma ~ N(alpha_T, Sigma ⊗ V_T)
        # Equivalently: A_j = alpha_T + L_V' Z L_S'  where L_V = chol(V_T), L_S = chol(Sigma)
        L_V  = np.linalg.cholesky(V_T)       # (1+Kp) × (1+Kp)
        L_S  = np.linalg.cholesky(Sigma_j)   # K × K
        Z    = np.random.randn(1 + K * P, K) # standard normal draw
        A_j  = alpha_T + L_V @ Z @ L_S.T     # posterior draw of A

        # Step 3: compute IRFs using Cholesky identification
        IRF_draws[j] = irf_cholesky(A_j, Sigma_j, P, K, H)
    except (np.linalg.LinAlgError, ValueError):
        IRF_draws[j] = np.nan
        n_fail += 1

if n_fail:
    print(f'  Note: {n_fail} draws failed (non-PD Sigma); set to NaN')

# Posterior summaries for the MP shock (Cholesky: shock index = 2, Fed funds last)
MP_IDX = 2
irf_mp_draws = IRF_draws[:, :, :, MP_IDX]   # (N_DRAWS, H+1, K)

irf_median = np.nanmedian(irf_mp_draws, axis=0)        # posterior median
irf_lo68   = np.nanpercentile(irf_mp_draws, 16, axis=0)  # 68% CI lower
irf_hi68   = np.nanpercentile(irf_mp_draws, 84, axis=0)  # 68% CI upper
irf_lo90   = np.nanpercentile(irf_mp_draws,  5, axis=0)  # 90% CI lower
irf_hi90   = np.nanpercentile(irf_mp_draws, 95, axis=0)  # 90% CI upper

# OLS point-estimate IRFs for comparison
irf_ols = irf_cholesky(B_ols, Sigma_ols, P, K, H)[:, :, MP_IDX]

print(f'Done. Posterior draws: {N_DRAWS - n_fail} valid / {N_DRAWS}')
print(f'\nImpact effects (h=0):')
print(f'{"Variable":<25}  {"OLS":>8}  {"BVAR median":>12}  {"68% CI":>18}')
print('-' * 68)
for k, name in enumerate(VAR_NAMES):
    print(f'{name:<25}  {irf_ols[0,k]:>8.4f}  {irf_median[0,k]:>12.4f}  '
          f'[{irf_lo68[0,k]:>7.4f}, {irf_hi68[0,k]:>7.4f}]')

### Step 4 — Figures 5.3 – 5.6

In [ ]:
# ── Figure 5.3: BVAR IRFs with credible intervals ─────────────────────────────
# Textbook Figure 5.3: posterior median + 68% + 90% credible intervals.
# Note: credible intervals have a direct probabilistic interpretation:
# 'There is a 68% posterior probability that the true IRF lies in the shaded band.'
# This differs from frequentist confidence intervals, which are statement about
# repeated sampling, not about the probability of the true parameter value.

h_axis = np.arange(H + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure 5.3: BVAR Impulse Responses to Monetary Policy Shock\n'
    f'Minnesota prior (λ₁ = {LAMBDA1}), {N_DRAWS} posterior draws  |  '
    f'{first_q} – {last_q}',
    fontsize=12, fontweight='bold'
)

for k, (ax, name) in enumerate(zip(axes, VAR_NAMES)):
    # 90% band (lighter)
    ax.fill_between(h_axis, irf_lo90[:, k], irf_hi90[:, k],
                    alpha=0.18, color=C_BVAR, label='90% CI')
    # 68% band (darker)
    ax.fill_between(h_axis, irf_lo68[:, k], irf_hi68[:, k],
                    alpha=0.38, color=C_BVAR, label='68% CI')
    # Posterior median
    ax.plot(h_axis, irf_median[:, k], color=C_BVAR, linewidth=2.5,
            label='Posterior median')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Response of {name}')
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage points')
    ax.set_xlim(0, H)
    ax.legend(fontsize=8, loc='lower right')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig53_bvar_irfs.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

**What to look for:**
- **GDP growth** should turn negative after 1–2 quarters, reaching a trough around $h = 4$–$6$. The 68% band excludes zero at the trough if the MP transmission to output is precisely estimated.
- **Inflation** may show the **price puzzle** — inflation rising initially before falling. If the 68% band lies entirely above zero in the first 5 quarters, the puzzle is not a sampling artefact but a genuine feature of the 3-variable specification (insufficient information for the Fed's forward-looking behavior). See Section 5.15.8 for interpretation.
- **Federal Funds Rate** should show the shock itself (positive on impact) decaying toward zero over 8–12 quarters, consistent with the typical persistence of policy rate changes.

**Bayesian vs frequentist uncertainty:** The shaded regions are *credible intervals*, not confidence intervals. A 68% credible interval means: given the data and the Minnesota prior, there is 68% posterior probability that the true IRF lies in this band. No asymptotic approximation or bootstrap is needed — the bands emerge directly from the posterior simulation.

In [ ]:
# ── Figure 5.4: BVAR vs OLS comparison ───────────────────────────────────────
# The key question: does shrinkage change the economic interpretation?
# If OLS lies inside the BVAR credible interval, both approaches are consistent.
# Differences typically arise in the cross-variable coefficients (where shrinkage is strongest).

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure 5.4: BVAR vs OLS Estimation\n'
    f'Posterior median (red) + 68% CI vs OLS point estimate (blue dashed)  |  '
    f'{first_q} – {last_q}',
    fontsize=12, fontweight='bold'
)

for k, (ax, name) in enumerate(zip(axes, VAR_NAMES)):
    # 68% credible band
    ax.fill_between(h_axis, irf_lo68[:, k], irf_hi68[:, k],
                    alpha=0.30, color=C_BVAR, label='BVAR 68% CI')
    # BVAR posterior median
    ax.plot(h_axis, irf_median[:, k], color=C_BVAR, linewidth=2.5,
            label='BVAR median')
    # OLS point estimate (dashed blue)
    ax.plot(h_axis, irf_ols[:, k], color=C_OLS, linewidth=2.0,
            linestyle='--', label='OLS')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Response of {name}')
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage points')
    ax.set_xlim(0, H)
    ax.legend(fontsize=8, loc='lower right')

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig54_bvar_vs_ols.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

In [ ]:
# ── Figure 5.5: Prior sensitivity analysis ────────────────────────────────────
# Vary λ₁ across [0.05, 0.10, 0.20, 0.50, 1.00].
# As λ₁ → ∞, the prior becomes flat and BVAR → OLS.
# As λ₁ → 0, the prior dominates and all coefficients → random walk prior.
# A robust result should be qualitatively stable across this range.

print('Computing prior sensitivity (5 values of λ₁)...')

irf_sensitivity = {}   # dict: lambda1 → posterior median IRF (H+1, K)
N_SENS = 1000          # fewer draws for speed (increase for publication)

for lam in LAMBDA1_GRID:
    alpha_s, V_s, S_s, nu_s, _, _ = bvar_posterior(Y, P, lam, LAMBDA3)
    irfs_s = np.zeros((N_SENS, H + 1, K))
    n_fail_s = 0
    for j in range(N_SENS):
        try:
            Sigma_j = draw_iw(nu_s, S_s)
            L_V = np.linalg.cholesky(V_s)
            L_S = np.linalg.cholesky(Sigma_j)
            Z   = np.random.randn(1 + K * P, K)
            A_j = alpha_s + L_V @ Z @ L_S.T
            irfs_s[j] = irf_cholesky(A_j, Sigma_j, P, K, H)[:, :, MP_IDX]
        except Exception:
            irfs_s[j] = np.nan
            n_fail_s += 1
    irf_sensitivity[lam] = np.nanmedian(irfs_s, axis=0)
    print(f'  λ₁ = {lam:5.2f}  → done ({n_fail_s} failures)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure 5.5: Prior Sensitivity — Effect of Overall Tightness λ₁\n'
    f'OLS shown for reference (dashed black)  |  {first_q} – {last_q}',
    fontsize=12, fontweight='bold'
)

for k, (ax, name) in enumerate(zip(axes, VAR_NAMES)):
    for i, lam in enumerate(LAMBDA1_GRID):
        med = irf_sensitivity[lam]
        ax.plot(h_axis, med[:, k], color=COLORS_5[i], linewidth=2.0,
                label=f'λ₁ = {lam}')
    # OLS reference
    ax.plot(h_axis, irf_ols[:, k], color='black', linewidth=1.5,
            linestyle='--', label='OLS')
    ax.axhline(0, color='black', linewidth=0.6, linestyle=':')
    ax.set_title(f'Response of {name}')
    ax.set_xlabel('Quarters after shock')
    ax.set_ylabel('Percentage points')
    ax.set_xlim(0, H)
    ax.legend(fontsize=8, ncol=2)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig55_prior_sensitivity.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

**What to look for in sensitivity analysis:** Three questions from the textbook:
1. **As λ₁ → ∞, does BVAR → OLS?** Yes: with a flat prior, the likelihood dominates. The λ₁=1.0 line should nearly coincide with the OLS dashed line.
2. **Which features are robust?** The qualitative shape — GDP declining, Fed funds persisting, price puzzle present — should appear for all λ₁ values. Magnitude sensitivity is acceptable; sign reversals are a red flag.
3. **Does the price puzzle persist?** If inflation stays positive at h=1,2,3 for all λ₁, the puzzle is not a prior artifact — it reflects genuinely insufficient information in the 3-variable system.

In [ ]:
# ── Figure 5.6: Posterior distributions of impact effects ────────────────────
# For GDP growth and inflation, Cholesky imposes zero impact by construction:
# the triangular identification means the Fed funds shock cannot move variables
# ordered before it on impact. So their h=0 distributions are degenerate at zero.
# Only the Fed funds impact effect has a non-degenerate posterior distribution.
# This visualises parameter uncertainty — the width reflects prior+data precision.

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle(
    f'Figure 5.6: Posterior Distributions of Impact Effects (h = 0)\n'
    f'Histogram of {N_DRAWS} posterior draws  |  λ₁ = {LAMBDA1}  |  {first_q} – {last_q}',
    fontsize=12, fontweight='bold'
)

for k, (ax, name) in enumerate(zip(axes, VAR_NAMES)):
    impact_draws = irf_mp_draws[:, 0, k]   # all posterior draws at h=0
    impact_draws = impact_draws[~np.isnan(impact_draws)]

    ax.hist(impact_draws, bins=50, color=C_BVAR, alpha=0.70,
            edgecolor='white', linewidth=0.4, density=True)

    # Vertical lines: posterior mean (red), median (blue dashed), OLS (black dotted)
    ax.axvline(impact_draws.mean(), color='red',   linewidth=1.8,
               linestyle='-',  label=f'Post. mean = {impact_draws.mean():.3f}')
    ax.axvline(np.median(impact_draws), color='blue', linewidth=1.8,
               linestyle='--', label=f'Post. median = {np.median(impact_draws):.3f}')
    ax.axvline(irf_ols[0, k], color='black', linewidth=1.8,
               linestyle=':', label=f'OLS = {irf_ols[0,k]:.3f}')

    ax.set_title(f'{name}\nImpact effect at $h=0$')
    ax.set_xlabel('Percentage points')
    ax.set_ylabel('Posterior density')
    ax.legend(fontsize=8)

fig.tight_layout()
path = os.path.join(OUT_DIR, 'fig56_posterior_impact.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show();  print(f'Saved: {path}')

**What to look for:** GDP growth and inflation show narrow distributions concentrated near zero — this is by Cholesky construction, not because the data say so. The Fed funds rate distribution is approximately Normal (as expected from the Normal-Inverse Wishart posterior). The OLS estimate (black dotted) should lie near the centre of the posterior distribution. If OLS falls in the tail of the posterior, the prior is dominating — reduce λ₁ or check the data.

---
## Summary

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 70)
print('CHAPTER 5 — BAYESIAN ESTIMATION: SUMMARY')
print('=' * 70)

print('\n--- Figure 5.2: MCMC Diagnostics ---')
print(f'Well-tuned   (c=1.70): AR = {ar_good:.1%}, ESS = {ess_good:.0f}, '
      f'Geweke Z = {z_good:.2f} ({"PASS" if abs(z_good) < 1.96 else "FAIL"})')
print(f'Poorly-tuned (c=0.05): AR = {ar_bad:.1%}, ESS = {ess_bad:.0f}, '
      f'Geweke Z = {z_bad:.2f} ({"PASS" if abs(z_bad) < 1.96 else "FAIL"})')
print(f'Efficiency ratio: {ess_good/ess_bad:.0f}× more effective samples per iteration (well vs poorly tuned)')

print('\n--- Section 5.15: BVAR Results (λ₁ = 0.20) ---')
print(f'Sample: {first_q} – {last_q}  |  VAR(4)  |  3 variables')
print(f'Posterior draws: {N_DRAWS} (conjugate NIW — no MCMC needed)')
print(f'\nOwn lag-1 shrinkage (BVAR vs OLS):')
for k, name in enumerate(VAR_NAMES):
    b_ols  = B_ols[1 + k, k]
    b_bvar = alpha_T[1 + k, k]
    print(f'  {name:<25}: OLS = {b_ols:.4f}, BVAR = {b_bvar:.4f}, '
          f'shrinkage = {100*abs(b_bvar-b_ols)/(abs(b_ols)+1e-10):.1f}%')

print('\n--- Price Puzzle Check ---')
puzzle_ols  = np.where(irf_ols[:12, 1] > 0)[0]
puzzle_bvar = np.where(irf_median[:12, 1] > 0)[0]
print(f'OLS: inflation > 0 at quarters:  {list(puzzle_ols)}')
print(f'BVAR median: inflation > 0 at:   {list(puzzle_bvar)}')
puzzle_68 = np.where(irf_lo68[:12, 1] > 0)[0]
print(f'Price puzzle robust (68% CI entirely above 0) at: {list(puzzle_68)}')
print('\n→ See Figure 5.3 inflation panel and Section 5.15.8 for interpretation.')

---
## Exercises

**Exercise 5.1 (Prior informativeness):** In Figure 5.1, the IG(1,1) prior for $\sigma^2$ has a very heavy right tail. Compute the prior probability that $\sigma^2 > 3$ for IG(1,1) vs IG(8,3). Which prior would be more appropriate if you believe the residual variance is around 0.5? What happens to the BVAR posterior if you inadvertently use a very diffuse IG prior for a low-variance variable?

**Exercise 5.2 (MCMC tuning):** Re-run `mcmc_generate_chains.py` with $c \in \{0.1, 0.5, 1.0, 1.7, 3.0\}$. Plot acceptance rates and ESS values against $c$. Find the optimal $c$ for the bivariate $\mathcal{N}(0, \Sigma_{\rho=0.7})$ target. Does the optimal $c$ change if $\rho = 0.3$ instead of $\rho = 0.7$?

**Exercise 5.3 (Prior derivation):** Show analytically that the posterior mean in the Normal-Normal model is a precision-weighted average of the prior mean and the OLS estimate: $\beta_T = (V_0^{-1} + X'X)^{-1}(V_0^{-1} \beta_0 + X'y)$. As $V_0^{-1} \to 0$ (diffuse prior), what does $\beta_T$ converge to? As $V_0^{-1} \to \infty$ (tight prior), what happens?

**Exercise 5.4 (Lag decay):** In the `bvar_posterior` function, the tightness at lag $l$ is `lambda1 * l^lambda3`. With `lambda3 = 1`, how much tighter is the prior on lag-4 coefficients vs lag-1 coefficients? Try `lambda3 = 0` (no decay) and `lambda3 = 2` (faster decay). Do the IRFs change substantially? What is the economic rationale for including lag decay?

**Exercise 5.5 (Pandemic priors):** Extend the sample to `2023-12-31` and re-estimate the BVAR. Do the IRFs change? To implement a simplified pandemic prior, add indicator dummies for 2020:Q1–2020:Q2 to $X_d$ with a large weight (e.g., $\phi = 0.01$). Does this recover the pre-pandemic dynamics? Compare with the pre-2020 sample estimate.

**Exercise 5.6 (Large BVAR):** Import the FRED-QD dataset from Chapter 2 and add `PCECC96` (real consumption) and `DPIC96` (real disposable income) to the 3-variable system. Re-estimate with the same Minnesota prior. Does the price puzzle weaken when more information is available? What happens to the GDP response? This exercises the "large information set" motivation for BVAR discussed in Section 5.7.

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 5: Bayesian Estimation | Last updated: March 2026  
Figures 5.1 – 5.6  |  Data: GDPC1, GDPDEF, FEDFUNDS (FRED)  |  MCMC target: $\mathcal{N}(\mathbf{0}, \Sigma_{\rho=0.7})$